# Module 6: Multi-Agent (Optional) (15 min)

> **Optional module.** You've already built and deployed a complete agent in Modules 1–5. This module adds delegation on top of that same agent.

Add agent delegation — when the customer service agent encounters a technical issue, it escalates to a **tech support specialist** agent. This is the agents-as-tools pattern: one orchestrator calls specialists like functions.

**Prerequisites:** Modules 1-4 completed

In [ ]:
!pip install -q -r requirements.txt

---

## Part 1: Build the Tech Support Specialist

A focused agent with its own tools and system prompt. It handles device troubleshooting, connectivity issues, and firmware updates.

In [2]:
from strands import Agent, tool
from customer_service_tools import lookup_customer, get_order_history, process_refund
from IPython.display import Markdown, display

# --- Tech support tools ---

@tool
def check_device_compatibility(device: str, issue: str) -> str:
    """Check if a device has known compatibility issues.

    Args:
        device: The device name or model
        issue: Description of the issue
    """
    known_issues = {
        "Wireless Headphones": "Known Bluetooth 5.0 pairing issue with older devices. Fix: Reset headphones (hold power 10s), then re-pair.",
        "USB-C Hub": "Some laptops require USB-C alt mode. Check laptop specs for DisplayPort over USB-C support.",
        "Mechanical Keyboard": "Firmware v2.1 has a key ghosting bug. Update to v2.3 via manufacturer website.",
    }
    for device_name, fix in known_issues.items():
        if device_name.lower() in device.lower():
            return f"Known issue found for {device_name}: {fix}"
    return f"No known issues found for '{device}'. Recommend standard troubleshooting: restart device, check connections, update drivers."


@tool
def run_diagnostic(device: str) -> str:
    """Run a remote diagnostic check on a device.

    Args:
        device: The device name or model to diagnose
    """
    return (
        f"Diagnostic results for {device}:\n"
        f"- Firmware: v2.1 (update available: v2.3)\n"
        f"- Connection: Stable\n"
        f"- Battery: 85%\n"
        f"- Last sync: 2 hours ago\n"
        f"Recommendation: Update firmware to resolve known issues."
    )


print("✅ Tech support tools defined")

✅ Tech support tools defined


---

## Part 2: Wrap the Specialist as a Tool

The `@tool` decorator turns the specialist agent into a callable tool for the orchestrator. The orchestrator decides when to delegate.

In [3]:
@tool
def tech_support_specialist(issue_description: str) -> str:
    """Escalate a technical issue to the tech support specialist agent.
    Use this when a customer has a device problem, connectivity issue,
    or needs technical troubleshooting beyond basic order/account help.

    Args:
        issue_description: Detailed description of the technical issue including device name and symptoms
    """
    specialist = Agent(
        tools=[check_device_compatibility, run_diagnostic],
        system_prompt="""You are a tech support specialist for an electronics store.
You diagnose device issues, check compatibility, and provide step-by-step fixes.
Be technical but clear. Always provide actionable next steps.""",
        callback_handler=None,  # Silent — don't stream to user
    )

    print(f"\n[DELEGATION] 🔧 Tech support specialist activated")
    print(f"[DELEGATION] 📋 Issue: {issue_description[:80]}...")

    response = specialist(issue_description)

    print(f"[DELEGATION] ✅ Specialist responded")
    return str(response)


print("✅ tech_support_specialist tool defined")

✅ tech_support_specialist tool defined


---

## Part 3: The Orchestrator Agent

The customer service agent now has `tech_support_specialist` as one of its tools. It decides when to escalate.

In [4]:
SYSTEM_PROMPT = """You are a customer service agent for an online electronics store.
Be helpful, professional, and concise.

You handle:
- Account lookups and order status
- Refund processing
- Basic questions

For TECHNICAL issues (device problems, connectivity, firmware, troubleshooting),
delegate to the tech_support_specialist tool. Provide it with the device name
and a clear description of the issue.

After getting the specialist's response, relay the solution to the customer
in a friendly, non-technical way."""

orchestrator = Agent(
    tools=[lookup_customer, get_order_history, process_refund, tech_support_specialist],
    system_prompt=SYSTEM_PROMPT,
    callback_handler=None,
)

# This should trigger delegation to the tech support specialist
result = orchestrator(
    "I'm customer C-1001. The wireless headphones I bought aren't pairing "
    "with my phone. I've tried restarting them but nothing works."
)
display(Markdown(str(result)))


[DELEGATION] 🔧 Tech support specialist activated
[DELEGATION] 📋 Issue: Customer C-1001 has wireless headphones that won't pair with their phone. They h...
[DELEGATION] ✅ Specialist responded


Hi Sarah! I've pulled up your account — everything looks good on your end. 😊

Our tech support specialist would like to help troubleshoot your pairing issue, but they'll need a couple of details to get started:

1. **What model are your wireless headphones?** (e.g., Sony WH-1000XM5, Bose QC45, etc.)
2. **What phone model are you using?** (e.g., iPhone 15, Samsung Galaxy S24, etc.)

Once I have those, I'll get you a solution right away!


In [5]:
# This should NOT trigger delegation — it's a simple order question
orchestrator = Agent(
    tools=[lookup_customer, get_order_history, process_refund, tech_support_specialist],
    system_prompt=SYSTEM_PROMPT,
    callback_handler=None,
)

result = orchestrator("I'm customer C-1002. Where is my keyboard order?")
display(Markdown(str(result)))

Hi Mike! Here's the update on your order:

- **Order:** ORD-5390 — Mechanical Keyboard ($149.99)
- **Status:** ⚠️ **Delayed**
- **Ordered:** April 15, 2025
- **Estimated Delivery:** April 25, 2025
- **Tracking Number:** TRK-776655

It looks like your keyboard is experiencing a shipping delay. I'm sorry about that! You can use the tracking number **TRK-776655** to get the latest updates from the carrier.

Is there anything else I can help you with, or would you like to discuss options like a refund?


---

## 🎯 Try It Yourself

Try different scenarios to see when the orchestrator delegates vs handles directly:

In [9]:
# Try these:
# "My USB-C hub isn't showing my external monitor" → should delegate
# "I want a refund for order ORD-5521" → should handle directly
# "The keyboard keys are sticking and some don't register" → should delegate

orchestrator = Agent(
    tools=[lookup_customer, get_order_history, process_refund, tech_support_specialist],
    system_prompt=SYSTEM_PROMPT,
    callback_handler=None,
)

#result = orchestrator("I'm C-1002. The mechanical keyboard keys are ghosting — I press one key and two characters appear.")
result = orchestrator("The keyboard keys are sticking and some don't register. The model is a Mechanical Keyboard v2.1. I've tried cleaning it but the problem persists. My customer ID is C-1002.")
display(Markdown(str(result)))


[DELEGATION] 🔧 Tech support specialist activated
[DELEGATION] 📋 Issue: Customer has a Mechanical Keyboard v2.1 with sticking keys and some keys not reg...
[DELEGATION] ✅ Specialist responded


Hi **Mike**! I've pulled up your account and connected with our tech specialist. Great news — your issue has been identified! Here's what you need to know:

---

### 🎉 Good News: It's a Software Fix, Not Hardware!

Your Mechanical Keyboard v2.1 has a **known firmware bug** that causes exactly what you're experiencing — sticking keys and missed keystrokes. That's why cleaning didn't help; it was never a physical problem!

---

### 🛠️ Here's How to Fix It:

1. **Go to the manufacturer's website** and find the **Support/Downloads** section for your keyboard model.
2. **Download firmware version v2.3** (the updated version that fixes this bug).
3. **Plug the keyboard in via USB** before starting the update.
4. **Run the updater** and follow the on-screen steps — just make sure not to unplug it during the process!
5. Once done, **restart the keyboard** and test your keys.

---

### ⚠️ A Few Tips:
- Use a **USB connection** (not wireless) for the update to ensure stability.
- If the problem **still persists after updating**, the switches may need a deeper inspection — but let's try this first!

---

Give that firmware update a try, Mike, and let me know how it goes! Is there anything else I can help you with? 😊


---

## 💬 Want a real multi-turn conversation?

In a notebook, each cell is a **single turn**. To chat back and forth with the orchestrator — which delegates technical issues to the specialist on any turn — run the companion script in a **terminal**. From the cloned repo:

```bash
cd samples/06-multi-agent
pip install -r requirements.txt
python chat.py
```

Type your messages, and `quit` (or Ctrl+C) to exit. The orchestrator keeps its context across turns and routes each message to the specialist or handles it directly.

---

## What's Next

The agent can now delegate to specialists. But how do you know it's working correctly at scale? In **Module 7: Evals**, you'll write automated evaluations to test the agent's behavior.